# Exploração de chaves e relacionamentos

Consultas pequenas para testar chaves candidatas e relações observadas nos notebooks 00 e 01. Não percorre histórico completo. O catálogo de rotas e downloads está em `docs/catalogo_recursos.csv`; as evidências consolidadas ficam em `docs/chaves_relacionamentos.md`.

Os caminhos de API podem mudar. Se receber 429/503 no Senado, pare e aguarde; não repita em loop.


In [1]:
%pip install requests pandas


Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import requests
import pandas as pd

CAMARA = 'https://dadosabertos.camara.leg.br/api/v2'
SENADO = 'https://legis.senado.leg.br'
session = requests.Session()
session.headers.update({'Accept': 'application/json', 'User-Agent': 'exploracao-painel-legislativo/0.1'})

def get_json(url, params=None):
    # Pausa conservadora para não exceder o limite divulgado pelo Senado.
    time.sleep(0.15)
    response = session.get(url, params=params, timeout=45)
    print(response.status_code, response.url, 'bytes=', len(response.content))
    response.raise_for_status()
    return response.json()

def key_profile(frame, columns):
    rows = []
    for column in columns:
        if column not in frame.columns:
            continue
        values = frame[column]
        rows.append({'field': column, 'rows': len(frame), 'nulls': int(values.isna().sum()),
                     'distinct_non_null': int(values.nunique(dropna=True)),
                     'duplicate_rows': int(values.duplicated(keep=False).sum())})
    return pd.DataFrame(rows)


## Câmara: proposição → autores, tramitações e votações

A proposição `117992` é um ID real observado no primeiro notebook. O perfil abaixo mede duplicidade apenas dentro da amostra retornada; não prova unicidade global.


In [3]:
proposal_id = 117992
proposal = get_json(f'{CAMARA}/proposicoes/{proposal_id}')['dados']
authors = get_json(f'{CAMARA}/proposicoes/{proposal_id}/autores')['dados']
steps = get_json(f'{CAMARA}/proposicoes/{proposal_id}/tramitacoes')['dados']
proposal_votes = get_json(f'{CAMARA}/proposicoes/{proposal_id}/votacoes')['dados']
print('proposal id:', proposal.get('id'), 'authors:', len(authors), 'tramitation rows:', len(steps), 'votation rows:', len(proposal_votes))
display(pd.json_normalize(authors).head())
display(pd.json_normalize(steps).head())


200 https://dadosabertos.camara.leg.br/api/v2/proposicoes/117992 bytes= 1757
200 https://dadosabertos.camara.leg.br/api/v2/proposicoes/117992/autores bytes= 278
200 https://dadosabertos.camara.leg.br/api/v2/proposicoes/117992/tramitacoes bytes= 11007
200 https://dadosabertos.camara.leg.br/api/v2/proposicoes/117992/votacoes bytes= 116
proposal id: 117992 authors: 1 tramitation rows: 17 votation rows: 0


,uri,nome,codTipo,tipo,ordemAssinatura,proponente
0,https://dadosabertos.camara.leg.br/api/v2/depu...,Rogério Silva,10000,Deputado(a),1,1


,dataHora,sequencia,siglaOrgao,uriOrgao,uriUltimoRelator,regime,descricaoTramitacao,codTipoTramitacao,descricaoSituacao,codSituacao,despacho,url,ambito,apreciacao
0,2003-06-02T19:05,1,PLEN,https://dadosabertos.camara.leg.br/api/v2/orga...,None,"Prioridade (Art. 151, II, RICD)",Apresentação de Proposição,100,Tramitando em Conjunto,925,Apresentação do Projeto de Lei pelo Deputado R...,https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
1,2003-06-12T00:00,11,MESA,https://dadosabertos.camara.leg.br/api/v2/orga...,None,"Prioridade (Art. 151, II, RICD)",Despacho de Apensação,129,Tramitando em Conjunto,925,Apense-se ao PL-7277/2002.(DESPACHO INICIAL),https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
2,2003-06-17T00:00,17,CCP,https://dadosabertos.camara.leg.br/api/v2/orga...,None,"Prioridade (Art. 151, II, RICD)",Publicação de Proposição,604,Tramitando em Conjunto,925,Encaminhada à publicação. Publicação Inicial n...,https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
3,2007-01-31T00:00,21,MESA,https://dadosabertos.camara.leg.br/api/v2/orga...,None,"Prioridade (Art. 151, II, RICD)",Arquivamento,502,Tramitando em Conjunto,925,Arquivado nos termos do Artigo 105 do Regiment...,https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
4,2007-02-13T14:01,22,MESA,https://dadosabertos.camara.leg.br/api/v2/orga...,None,"Prioridade (Art. 151, II, RICD)",Desarquivamento a Pedido,640,Tramitando em Conjunto,925,"Apresentação do REQUERIMENTO N.º 219, DE 2007,...",https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário


## Senado: parlamentar → mandato/filiação; votação → processo e voto

A votação fica filtrada a um parlamentar e a um mês. Evite a listagem ampla de processos e de votações, que produziu respostas grandes nos primeiros experimentos.


In [4]:
senator_code = '5672'
senator = get_json(f'{SENADO}/dadosabertos/senador/{senator_code}.json')
mandates = get_json(f'{SENADO}/dadosabertos/senador/{senator_code}/mandatos.json')
affiliations = get_json(f'{SENADO}/dadosabertos/senador/{senator_code}/filiacoes.json')
print('Senador detail envelopes:', list(senator), list(mandates), list(affiliations))
print('Parlamentar keys:', list(senator['DetalheParlamentar']['Parlamentar']))
print('Mandatos keys:', list(mandates['MandatoParlamentar']['Parlamentar']))
print('Filiações keys:', list(affiliations['FiliacaoParlamentar']['Parlamentar']))


200 https://legis.senado.leg.br/dadosabertos/senador/5672.json bytes= 3618
200 https://legis.senado.leg.br/dadosabertos/senador/5672/mandatos.json bytes= 1244
200 https://legis.senado.leg.br/dadosabertos/senador/5672/filiacoes.json bytes= 978
Senador detail envelopes: ['DetalheParlamentar'] ['MandatoParlamentar'] ['FiliacaoParlamentar']
Parlamentar keys: ['IdentificacaoParlamentar', 'DadosBasicosParlamentar', 'Telefones', 'OutrasInformacoes']
Mandatos keys: ['Codigo', 'Nome', 'Mandatos']
Filiações keys: ['Codigo', 'Nome', 'Filiacoes']


In [5]:
voting = get_json(f'{SENADO}/dadosabertos/votacao.json', {
    'codigoParlamentar': senator_code,
    'dataInicio': '2023-02-01',
    'dataFim': '2023-02-28',
})
print('votations in bounded sample:', len(voting))
if voting:
    vote = voting[0]
    print({key: vote.get(key) for key in ['idProcesso', 'codigoMateria', 'codigoSessao', 'codigoSessaoVotacao']})
    print('nested votes:', len(vote.get('votos', [])))
    display(pd.json_normalize(vote.get('votos', [])))
    process = get_json(f"{SENADO}/dadosabertos/processo/{vote['idProcesso']}.json")
    print('process keys:', list(process))
    print('process id matches vote idProcesso:', process.get('id') == vote.get('idProcesso'))
    print('process codigoMateria matches vote codigoMateria:', process.get('codigoMateria') == vote.get('codigoMateria'))


200 https://legis.senado.leg.br/dadosabertos/votacao.json?codigoParlamentar=5672&dataInicio=2023-02-01&dataFim=2023-02-28 bytes= 1666
votations in bounded sample: 1
{'idProcesso': 8361684, 'codigoMateria': 155773, 'codigoSessao': 320695, 'codigoSessaoVotacao': 6674}
nested votes: 1


,codigoParlamentar,descricaoVotoParlamentar,nomeParlamentar,sexoParlamentar,siglaPartidoParlamentar,siglaUFParlamentar,siglaVotoParlamentar
0,5672,None,Alan Rick,M,UNIÃO,AC,Votou


200 https://legis.senado.leg.br/dadosabertos/processo/8361684.json bytes= 12478
process keys: ['id', 'codigoMateria', 'identificacao', 'sigla', 'descricaoSigla', 'numero', 'ano', 'objetivo', 'casaIdentificadora', 'siglaEnteIdentificador', 'identificacaoExterna', 'conteudo', 'documento', 'tramitando', 'dataInicioEfetivo', 'situacaoAtual', 'siglaSituacaoAtual', 'dataSituacaoAtual', 'classificacoes', 'autoriaIniciativa', 'idProcessoCasaInicial', 'identificacaoProcessoInicial', 'siglaCasaIniciadora', 'comissaoMpv', 'outrosNumeros', 'autuacoes', 'deliberacao', 'normaGerada']
process id matches vote idProcesso: True
process codigoMateria matches vote codigoMateria: True


## Critério para fechar uma chave

Registre chave como **confirmada** só quando o contrato/documentação ou verificações de unicidade e detalhe a sustentarem. Guarde os códigos brutos (incluindo texto vs. número) e não una Câmara e Senado por nome, ementa ou número isolado. Anote divergências e chaves não confirmadas em `docs/chaves_relacionamentos.md`.
